# Kimi Researcher — Experiment Analysis

Analyze all Kimi experiment results across CPU and GPU platforms, 3 trials per seed.

In [ ]:
import json
import glob
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from collections import defaultdict
from pathlib import Path

matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['font.size'] = 11

BASE = Path('..') if Path('../results').exists() else Path('.')
print(f'Base: {BASE.resolve()}')

## 1. Data Collection

In [ ]:
rows = []

# results/ directory (organized trials)
for seed_dir in sorted((BASE / 'results' / 'kimi').iterdir()):
    if not seed_dir.is_dir(): continue
    seed_name = seed_dir.name
    platform = 'gpu' if '_gpu' in seed_name else 'cpu'
    seed = seed_name.replace('_gpu','').replace('_cpu','').replace('_',' ')
    
    for trial_dir in sorted(seed_dir.iterdir()):
        if not trial_dir.is_dir() or not trial_dir.name.startswith('trial'): continue
        trial_num = int(trial_dir.name.replace('trial',''))
        
        review_files = sorted(trial_dir.glob('code/idea_*/reviews.json'))
        if not review_files: continue
        try:
            reviews = json.loads(review_files[-1].read_text())
        except: continue
        
        row = {'seed': seed, 'platform': platform, 'trial': trial_num,
               'avg_score': reviews.get('avg_score', 0)}
        
        for rev in reviews.get('reviews', []):
            name = rev.get('source', '?')
            short = name.replace('agent:', '')
            row[f'reviewer_{short}'] = rev.get('overall_score')
            for dim, val in rev.get('scores', {}).items():
                if isinstance(val, (int, float)):
                    row[f'dim_{dim}_{short}'] = val
        
        # Tracker
        tracker_file = trial_dir / 'tracker' / 'tracker.json'
        if tracker_file.exists():
            try:
                tracker = json.loads(tracker_file.read_text())
                actions = tracker.get('actions', [])
                for a in actions:
                    stage = a.get('stage', '')
                    if 'self_review' in stage:
                        key = f'sr_{stage}_count'
                        row[key] = row.get(key, 0) + 1
                        if a.get('outcome') == 'success':
                            row[f'sr_{stage}_pass'] = row.get(f'sr_{stage}_pass', 0) + 1
                    elapsed = a.get('elapsed_seconds', 0)
                    if elapsed > 0:
                        key = f'time_{stage}'
                        row[key] = row.get(key, 0) + elapsed
            except: pass
        
        summary_file = trial_dir / 'tracker' / 'summary.json'
        if summary_file.exists():
            try:
                s = json.loads(summary_file.read_text())
                row['wall_time_h'] = s.get('wall_time_seconds', 0) / 3600
                row['total_steps'] = s.get('total_steps', 0)
                row['ideas_tried'] = s.get('ideas_tried', 0)
            except: pass
        
        rows.append(row)

# outputs/kimi/run_N/ (GPU results not yet in results/)
for run_dir in sorted((BASE / 'outputs' / 'kimi').glob('run_*')):
    run_num = int(run_dir.name.split('_')[1])
    for seed_dir in sorted(run_dir.iterdir()):
        if not seed_dir.is_dir(): continue
        seed = seed_dir.name.replace('_', ' ')
        dup = any(r['seed']==seed and r['trial']==run_num and r['platform']=='gpu' for r in rows)
        if dup: continue
        
        review_files = sorted(seed_dir.glob('idea_*/reviews.json'))
        if not review_files: continue
        try:
            reviews = json.loads(review_files[-1].read_text())
        except: continue
        
        row = {'seed': seed, 'platform': 'gpu', 'trial': run_num,
               'avg_score': reviews.get('avg_score', 0)}
        for rev in reviews.get('reviews', []):
            name = rev.get('source', '?').replace('agent:', '')
            row[f'reviewer_{name}'] = rev.get('overall_score')
            for dim, val in rev.get('scores', {}).items():
                if isinstance(val, (int, float)):
                    row[f'dim_{dim}_{name}'] = val
        
        tracker_file = seed_dir / 'tracker.json'
        if tracker_file.exists():
            try:
                tracker = json.loads(tracker_file.read_text())
                for a in tracker.get('actions', []):
                    stage = a.get('stage', '')
                    if 'self_review' in stage:
                        key = f'sr_{stage}_count'
                        row[key] = row.get(key, 0) + 1
                        if a.get('outcome') == 'success':
                            row[f'sr_{stage}_pass'] = row.get(f'sr_{stage}_pass', 0) + 1
                    elapsed = a.get('elapsed_seconds', 0)
                    if elapsed > 0:
                        row[f'time_{stage}'] = row.get(f'time_{stage}', 0) + elapsed
            except: pass
        
        summary_file = seed_dir / 'summary.json'
        if summary_file.exists():
            try:
                s = json.loads(summary_file.read_text())
                row['wall_time_h'] = s.get('wall_time_seconds', 0) / 3600
                row['total_steps'] = s.get('total_steps', 0)
                row['ideas_tried'] = s.get('ideas_tried', 0)
            except: pass
        rows.append(row)

# Timestamp-based outputs
for ws in sorted((BASE / 'outputs' / 'kimi').iterdir()):
    if not ws.is_dir() or ws.name.startswith('run_') or ws.name == 'logs': continue
    parts = ws.name.rsplit('_', 2)
    if len(parts) < 3 or not parts[-2].isdigit(): continue
    seed = '_'.join(parts[:-2]).replace('_', ' ')
    dup = any(r['seed']==seed and r['trial']==1 and r['platform']=='gpu' for r in rows)
    if dup: continue
    review_files = sorted(ws.glob('idea_*/reviews.json'))
    if not review_files: continue
    try:
        reviews = json.loads(review_files[-1].read_text())
    except: continue
    row = {'seed': seed, 'platform': 'gpu', 'trial': 1,
           'avg_score': reviews.get('avg_score', 0)}
    for rev in reviews.get('reviews', []):
        name = rev.get('source', '?').replace('agent:', '')
        row[f'reviewer_{name}'] = rev.get('overall_score')
    rows.append(row)

df = pd.DataFrame(rows)
print(f'Total trials: {len(df)}')
print(f'CPU: {len(df[df.platform=="cpu"])}, GPU: {len(df[df.platform=="gpu"])}')
df.head()

## 2. Peer Review Scores by Seed

In [ ]:
summary = df.groupby(['platform', 'seed'])['avg_score'].agg(['mean', 'std', 'count']).round(2)
summary['se'] = (summary['std'] / np.sqrt(summary['count'])).round(2)
print(summary.to_string())

print(f"\nOverall CPU: {df[df.platform=='cpu']['avg_score'].mean():.2f} ± {df[df.platform=='cpu']['avg_score'].sem():.2f}")
print(f"Overall GPU: {df[df.platform=='gpu']['avg_score'].mean():.2f} ± {df[df.platform=='gpu']['avg_score'].sem():.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for ax, platform in zip(axes, ['cpu', 'gpu']):
    pdata = df[df.platform == platform]
    seeds = sorted(pdata.seed.unique())
    means = [pdata[pdata.seed == s]['avg_score'].mean() for s in seeds]
    sems = [pdata[pdata.seed == s]['avg_score'].sem() for s in seeds]
    
    colors = ['#2196F3' if m >= 4 else '#F44336' for m in means]
    bars = ax.barh(range(len(seeds)), means, xerr=sems, color=colors, alpha=0.8, capsize=4)
    ax.set_yticks(range(len(seeds)))
    ax.set_yticklabels([s.replace(' ', '\n') if len(s) > 20 else s for s in seeds], fontsize=9)
    ax.set_xlabel('Peer Review Score')
    ax.set_title(f'Kimi — {platform.upper()} Seeds')
    ax.axvline(x=8, color='green', linestyle='--', alpha=0.5, label='Accept (8)')
    ax.axvline(x=5, color='orange', linestyle='--', alpha=0.5, label='Revision (5)')
    ax.set_xlim(0, 10)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('kimi_scores_by_seed.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Reviewer Bias Analysis

In [ ]:
reviewer_cols = [c for c in df.columns if c.startswith('reviewer_')]
reviewer_data = {}
for col in reviewer_cols:
    name = col.replace('reviewer_', '')
    scores = df[col].dropna()
    reviewer_data[name] = scores

fig, ax = plt.subplots(figsize=(10, 5))
positions = range(len(reviewer_data))
names = list(reviewer_data.keys())
bp = ax.boxplot([reviewer_data[n].values for n in names], positions=positions,
                patch_artist=True, widths=0.6)
colors = ['#2196F3', '#4CAF50', '#FF9800']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_xticks(positions)
ax.set_xticklabels(names)
ax.set_ylabel('Score')
ax.set_title('Reviewer Score Distributions (Kimi papers)')
ax.axhline(y=8, color='green', linestyle='--', alpha=0.3, label='Accept')
ax.axhline(y=4, color='red', linestyle='--', alpha=0.3, label='Reject')

for i, name in enumerate(names):
    scores = reviewer_data[name]
    ax.annotate(f'μ={scores.mean():.1f}', xy=(i, scores.mean()),
                xytext=(i+0.3, scores.mean()+0.3), fontsize=9)

ax.legend()
plt.tight_layout()
plt.savefig('kimi_reviewer_bias.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Per-Dimension Analysis

In [ ]:
dimensions = ['novelty', 'soundness', 'significance', 'clarity', 'reproducibility',
              'experimental_rigor', 'references', 'reference_integrity', 'results_integrity']

dim_means = {}
for dim in dimensions:
    cols = [c for c in df.columns if c.startswith(f'dim_{dim}_')]
    if cols:
        all_vals = pd.concat([df[c] for c in cols]).dropna()
        dim_means[dim] = all_vals.mean()

fig, ax = plt.subplots(figsize=(12, 5))
dims = list(dim_means.keys())
vals = [dim_means[d] for d in dims]
colors = ['#F44336' if v < 4 else '#FF9800' if v < 6 else '#4CAF50' for v in vals]
bars = ax.bar(range(len(dims)), vals, color=colors, alpha=0.8)
ax.set_xticks(range(len(dims)))
ax.set_xticklabels([d.replace('_', '\n') for d in dims], rotation=0, fontsize=9)
ax.set_ylabel('Mean Score (1-10)')
ax.set_title('Kimi Papers — Per-Dimension Scores (all reviewers)')
ax.axhline(y=6, color='green', linestyle='--', alpha=0.3)
ax.axhline(y=4, color='red', linestyle='--', alpha=0.3)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.1f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('kimi_dimension_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. CPU vs GPU Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

cpu_scores = df[df.platform == 'cpu']['avg_score']
gpu_scores = df[df.platform == 'gpu']['avg_score']

bp = ax.boxplot([cpu_scores.values, gpu_scores.values],
                labels=['CPU', 'GPU'], patch_artist=True, widths=0.5)
bp['boxes'][0].set_facecolor('#2196F3')
bp['boxes'][1].set_facecolor('#FF9800')
for box in bp['boxes']:
    box.set_alpha(0.7)

ax.set_ylabel('Peer Review Score')
ax.set_title('Kimi: CPU vs GPU Performance')
ax.axhline(y=8, color='green', linestyle='--', alpha=0.3, label='Accept')

ax.annotate(f'μ={cpu_scores.mean():.2f}±{cpu_scores.sem():.2f}',
            xy=(1, cpu_scores.mean()), xytext=(1.3, cpu_scores.mean()), fontsize=10)
ax.annotate(f'μ={gpu_scores.mean():.2f}±{gpu_scores.sem():.2f}',
            xy=(2, gpu_scores.mean()), xytext=(2.3, gpu_scores.mean()), fontsize=10)

plt.tight_layout()
plt.savefig('kimi_cpu_vs_gpu.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'CPU: {cpu_scores.mean():.2f} ± {cpu_scores.sem():.2f} (n={len(cpu_scores)})')
print(f'GPU: {gpu_scores.mean():.2f} ± {gpu_scores.sem():.2f} (n={len(gpu_scores)})')
print(f'Diff: {cpu_scores.mean() - gpu_scores.mean():+.2f}')

## 6. Self-Review Impact

In [ ]:
sr_cols = [c for c in df.columns if c.startswith('sr_') and c.endswith('_count')]
sr_data = []
for col in sr_cols:
    gate = col.replace('sr_', '').replace('_count', '')
    counts = df[col].dropna()
    pass_col = col.replace('_count', '_pass')
    passes = df[pass_col].dropna() if pass_col in df.columns else pd.Series(dtype=float)
    sr_data.append({
        'gate': gate,
        'total_invocations': int(counts.sum()),
        'total_passes': int(passes.sum()) if len(passes) > 0 else 0,
        'mean_per_run': counts.mean(),
    })

sr_df = pd.DataFrame(sr_data)
sr_df['pass_rate'] = (sr_df['total_passes'] / sr_df['total_invocations'] * 100).round(1)
sr_df['revisions_per_run'] = (sr_df['mean_per_run'] - sr_df['total_passes'] / len(df)).round(1)
print(sr_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
gates = sr_df['gate'].tolist()
x = range(len(gates))
ax.bar(x, sr_df['mean_per_run'], color='#2196F3', alpha=0.7, label='Avg invocations/run')
ax.set_xticks(x)
ax.set_xticklabels([g.replace('self_review_', '') for g in gates])
ax.set_ylabel('Count')
ax.set_title('Self-Review: Average Invocations Per Run')
for i, row in sr_df.iterrows():
    ax.text(i, row['mean_per_run'] + 0.1, f"{row['pass_rate']}% pass", ha='center', fontsize=9)
ax.legend()
plt.tight_layout()
plt.savefig('kimi_self_review_impact.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Time Analysis

In [ ]:
time_cols = [c for c in df.columns if c.startswith('time_')]
time_data = []
for col in time_cols:
    stage = col.replace('time_', '')
    vals = df[col].dropna() / 60  # convert to minutes
    if len(vals) > 0:
        time_data.append({
            'stage': stage,
            'n': len(vals),
            'mean_min': vals.mean(),
            'se_min': vals.sem(),
            'min_min': vals.min(),
            'max_min': vals.max(),
        })

time_df = pd.DataFrame(time_data).sort_values('mean_min', ascending=False)
print(time_df.round(1).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
stages = time_df['stage'].tolist()
means = time_df['mean_min'].tolist()
sems = time_df['se_min'].tolist()
colors = ['#F44336' if m > 60 else '#FF9800' if m > 10 else '#4CAF50' for m in means]
ax.barh(range(len(stages)), means, xerr=sems, color=colors, alpha=0.8, capsize=3)
ax.set_yticks(range(len(stages)))
ax.set_yticklabels(stages)
ax.set_xlabel('Time (minutes)')
ax.set_title('Kimi — Time Spent Per Stage (total across all invocations)')
plt.tight_layout()
plt.savefig('kimi_time_per_stage.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Wall Time Distribution

In [ ]:
wt = df[df['wall_time_h'] > 0]

fig, ax = plt.subplots(figsize=(10, 5))
for platform, color in [('cpu', '#2196F3'), ('gpu', '#FF9800')]:
    data = wt[wt.platform == platform]['wall_time_h']
    if len(data) > 0:
        ax.hist(data, bins=15, alpha=0.6, color=color, label=f'{platform.upper()} (μ={data.mean():.1f}h)')

ax.set_xlabel('Wall Time (hours)')
ax.set_ylabel('Count')
ax.set_title('Kimi — Wall Time Distribution Per Run')
ax.legend()
plt.tight_layout()
plt.savefig('kimi_wall_time.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Score vs Time Correlation

In [ ]:
wt_scores = df[(df['wall_time_h'] > 0) & (df['avg_score'] > 0)]

fig, ax = plt.subplots(figsize=(8, 6))
for platform, color, marker in [('cpu', '#2196F3', 'o'), ('gpu', '#FF9800', 's')]:
    data = wt_scores[wt_scores.platform == platform]
    ax.scatter(data['wall_time_h'], data['avg_score'], c=color, marker=marker,
               alpha=0.7, s=60, label=f'{platform.upper()} (n={len(data)})')

ax.set_xlabel('Wall Time (hours)')
ax.set_ylabel('Peer Review Score')
ax.set_title('Kimi — Score vs Wall Time')
ax.axhline(y=8, color='green', linestyle='--', alpha=0.3, label='Accept')
ax.legend()

# Correlation
if len(wt_scores) > 2:
    corr = wt_scores[['wall_time_h', 'avg_score']].corr().iloc[0, 1]
    ax.text(0.05, 0.95, f'r = {corr:.2f}', transform=ax.transAxes, fontsize=11)

plt.tight_layout()
plt.savefig('kimi_score_vs_time.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Trial-over-Trial Improvement

In [ ]:
trial_means = df.groupby(['platform', 'trial'])['avg_score'].agg(['mean', 'sem', 'count'])
print(trial_means.round(2))

fig, ax = plt.subplots(figsize=(8, 5))
for platform, color in [('cpu', '#2196F3'), ('gpu', '#FF9800')]:
    pdata = df[df.platform == platform]
    trials = sorted(pdata.trial.unique())
    means = [pdata[pdata.trial == t]['avg_score'].mean() for t in trials]
    sems = [pdata[pdata.trial == t]['avg_score'].sem() for t in trials]
    ax.errorbar(trials, means, yerr=sems, marker='o', capsize=5,
                color=color, label=f'{platform.upper()}')

ax.set_xlabel('Trial Number')
ax.set_ylabel('Mean Peer Review Score')
ax.set_title('Kimi — Score Across Trials')
ax.legend()
ax.set_xticks(sorted(df.trial.unique()))
plt.tight_layout()
plt.savefig('kimi_trial_improvement.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Summary Statistics

In [ ]:
print('=== KIMI RESEARCHER SUMMARY ===')
print(f'Total trials: {len(df)}')
print(f'  CPU: {len(df[df.platform=="cpu"])} trials across {df[df.platform=="cpu"].seed.nunique()} seeds')
print(f'  GPU: {len(df[df.platform=="gpu"])} trials across {df[df.platform=="gpu"].seed.nunique()} seeds')
print()
print(f'Overall mean score: {df.avg_score.mean():.2f} ± {df.avg_score.sem():.2f}')
print(f'  CPU: {df[df.platform=="cpu"].avg_score.mean():.2f} ± {df[df.platform=="cpu"].avg_score.sem():.2f}')
print(f'  GPU: {df[df.platform=="gpu"].avg_score.mean():.2f} ± {df[df.platform=="gpu"].avg_score.sem():.2f}')
print()
print(f'Best score: {df.avg_score.max():.1f}')
best = df[df.avg_score == df.avg_score.max()].iloc[0]
print(f'  Seed: {best.seed} ({best.platform}), Trial {best.trial}')
print()
print(f'Papers scoring >= 4: {len(df[df.avg_score >= 4])} / {len(df)} ({100*len(df[df.avg_score >= 4])/len(df):.0f}%)')
print(f'Papers scoring >= 5: {len(df[df.avg_score >= 5])} / {len(df)} ({100*len(df[df.avg_score >= 5])/len(df):.0f}%)')
print(f'Papers scoring >= 6: {len(df[df.avg_score >= 6])} / {len(df)} ({100*len(df[df.avg_score >= 6])/len(df):.0f}%)')

## 12. Self-Review Score Trends

How do self-review scores change across successive reviews within the same trial?

In [ ]:
import json as _json
from collections import defaultdict

# Collect self-review score sequences from tracker data
sequences = defaultdict(lambda: defaultdict(list))

for tracker_file in sorted((BASE / "results" / "kimi").glob("*/trial*/tracker/tracker.json")):
    try:
        tracker = _json.loads(tracker_file.read_text())
        trial_key = "/".join(tracker_file.parts[-4:-2])
        for a in tracker.get("actions", []):
            stage = a.get("stage", "")
            if "self_review" not in stage: continue
            details = a.get("details", "")
            if "score=" in details:
                score = float(details.split("score=")[1].split(",")[0])
                sequences[stage][trial_key].append(score)
    except: pass

for tracker_file in sorted((BASE / "outputs" / "kimi").glob("run_*/*/tracker.json")):
    try:
        tracker = _json.loads(tracker_file.read_text())
        parts = tracker_file.parts
        trial_key = f"{parts[-2]}_gpu/run_{parts[-3].split(chr(95))[1]}"
        for a in tracker.get("actions", []):
            stage = a.get("stage", "")
            if "self_review" not in stage: continue
            details = a.get("details", "")
            if "score=" in details:
                score = float(details.split("score=")[1].split(",")[0])
                sequences[stage][trial_key].append(score)
    except: pass

# Plot trends
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
gates = ["self_review_idea", "self_review_experiment", "self_review_paper"]
titles = ["Idea Self-Review", "Experiment Self-Review", "Paper Self-Review"]
colors = ["#2196F3", "#FF9800", "#4CAF50"]

for ax, gate, title, color in zip(axes, gates, titles, colors):
    all_seqs = sequences.get(gate, {})
    if not all_seqs:
        ax.set_title(f"{title} (no data)")
        continue
    max_len = max(len(s) for s in all_seqs.values())
    
    # Plot individual trajectories (light)
    for seq in all_seqs.values():
        ax.plot(range(1, len(seq)+1), seq, color=color, alpha=0.15, linewidth=1)
    
    # Plot mean trajectory (bold)
    means = []
    sems = []
    ns = []
    for pos in range(max_len):
        scores = [s[pos] for s in all_seqs.values() if pos < len(s)]
        if scores:
            means.append(np.mean(scores))
            sems.append(np.std(scores) / np.sqrt(len(scores)))
            ns.append(len(scores))
    
    positions = range(1, len(means)+1)
    ax.errorbar(positions, means, yerr=sems, color=color, linewidth=2.5,
                marker="o", capsize=4, label="Mean ± SE")
    ax.axhline(y=8, color="green", linestyle="--", alpha=0.3, label="Pass (8)")
    ax.axhline(y=6, color="orange", linestyle="--", alpha=0.3, label="Pass (6)")
    
    # Annotate N at each position
    for x, m, n in zip(positions, means, ns):
        ax.annotate(f"n={n}", xy=(x, m), xytext=(x, m-0.8), fontsize=8, ha="center")
    
    ax.set_xlabel("Review Number")
    ax.set_title(title)
    ax.set_ylim(0, 10)
    ax.legend(fontsize=8)

axes[0].set_ylabel("Self-Review Score")
fig.suptitle("Kimi — Self-Review Score Trends (individual trials + mean)", fontsize=13)
plt.tight_layout()
plt.savefig("kimi_self_review_trends.png", dpi=150, bbox_inches="tight")
plt.show()

# Print improvement stats
for gate, title in zip(gates, titles):
    all_seqs = sequences.get(gate, {})
    multi = [(k, s) for k, s in all_seqs.items() if len(s) >= 2]
    if not multi: continue
    improving = sum(1 for _, s in multi if s[-1] > s[0])
    declining = sum(1 for _, s in multi if s[-1] < s[0])
    flat = sum(1 for _, s in multi if s[-1] == s[0])
    print(f"{title}: {len(multi)} multi-review trials — "
          f"{improving} improving ({100*improving/len(multi):.0f}%), "
          f"{declining} declining ({100*declining/len(multi):.0f}%), "
          f"{flat} flat ({100*flat/len(multi):.0f}%)")